<a href="https://colab.research.google.com/github/yaseralomari333-stack/NLP-DSIMC/blob/main/NLP-DSIMC0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [2]:
# 2) تحميل الكود من GitHub على Colab فقط
!rm -rf /content/NLP-DSIMC
!git clone https://github.com/yaseralomari333-stack/NLP-DSIMC.git /content/NLP-DSIMC

%cd /content/NLP-DSIMC
!ls

Cloning into '/content/NLP-DSIMC'...
remote: Enumerating objects: 29, done.
remote: Counting objects: 100% (29/29), done.
remote: Compressing objects: 100% (28/28), done.
remote: Total 29 (delta 1), reused 22 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (29/29), 28.00 KiB | 895.00 KiB/s, done.
Resolving deltas: 100% (1/1), done.
/content/NLP-DSIMC
colab_run.ipynb  Models		   README.md	     RUN.py
DAL		 NLP-DSIMC0.ipynb  requirements.txt  _smoke_test.py


In [3]:
# 3) تثبيت المتطلبات
!pip install -r requirements.txt

In [4]:
# 4) نسخ الداتاست من Drive إلى Colab
# هيك الداتاست بتصير موجودة على Colab، مش بنشتغل مباشرة من Drive

import os, shutil

drive_dataset_path = "/content/drive/MyDrive/Arabic Suicidal Dataset/Arabic Suicidal Dataset.xlsx"

if not os.path.exists(drive_dataset_path):
    matches = []
    for root, dirs, files in os.walk("/content/drive/MyDrive"):
        for file in files:
            if "suicidal" in file.lower() and file.lower().endswith((".xlsx", ".csv")):
                matches.append(os.path.join(root, file))

    if not matches:
        raise FileNotFoundError("لم أجد ملف الداتاست في Google Drive")

    drive_dataset_path = matches[0]

local_data_dir = "/content/NLP-DSIMC/DAL"
os.makedirs(local_data_dir, exist_ok=True)

local_dataset_path = os.path.join(local_data_dir, os.path.basename(drive_dataset_path))
shutil.copy2(drive_dataset_path, local_dataset_path)

print("Dataset copied to Colab:")
print(local_dataset_path)

Dataset copied to Colab:
/content/NLP-DSIMC/DAL/Arabic Suicidal Dataset.xlsx


In [5]:
# 5) إنشاء مجلد نتائج على Colab فقط
import os

OUTPUT_DIR = "/content/outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Reports/results will be saved here:")
print(OUTPUT_DIR)

Reports/results will be saved here:
/content/outputs


In [6]:
# 6) إصلاح خطأ Trainer وتشغيل MARBERT ثم المقارنة

import os
from pathlib import Path

# مكان حفظ النتائج على Colab فقط
OUTPUT_DIR = "/content/outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

os.environ["NLP_ARTIFACTS_DIR"] = OUTPUT_DIR
%env NLP_ARTIFACTS_DIR=/content/outputs

# الدخول لمجلد المشروع
%cd /content/NLP-DSIMC

# إصلاح الخطأ:
# TypeError: Trainer.__init__() got an unexpected keyword argument 'tokenizer'
marbert_file = Path("/content/NLP-DSIMC/Models/MARBERTOps.py")
text = marbert_file.read_text(encoding="utf-8")

text = text.replace(
    "tokenizer=self.tokenizer,",
    "processing_class=self.tokenizer,"
)

marbert_file.write_text(text, encoding="utf-8")

print("تم إصلاح ملف MARBERTOps.py")

# تحديد مسار الداتاست
if "local_dataset_path" not in globals():
    local_dataset_path = "/content/NLP-DSIMC/DAL/Arabic Suicidal Dataset.xlsx"

print("Dataset path:")
print(local_dataset_path)

# لو baseline_metrics مش موجود، شغّل baseline أولاً
if not os.path.exists("/content/outputs/baseline_metrics.json"):
    print("baseline_metrics.json غير موجود، سيتم تشغيل Baseline أولاً...")
    !python RUN.py train_baseline --data "{local_dataset_path}"

# تشغيل MARBERT فقط
print("تشغيل MARBERT...")
!python RUN.py train_marbert --data "{local_dataset_path}" --fp16

# تشغيل المقارنة
print("تشغيل المقارنة...")
!python RUN.py compare --data "{local_dataset_path}"

print("انتهى التشغيل. النتائج محفوظة في:")
print(OUTPUT_DIR)

env: NLP_ARTIFACTS_DIR=/content/outputs
/content/NLP-DSIMC
تم إصلاح ملف MARBERTOps.py
Dataset path:
/content/NLP-DSIMC/DAL/Arabic Suicidal Dataset.xlsx
تشغيل MARBERT...
Train=4115  Val=458  Test=1144  Classes=['not', 's']
Loading weights: 100% 199/199 [00:00<00:00, 1437.44it/s, Materializing param=bert.pooler.dense.weight]
BertForSequenceClassification LOAD REPORT from: UBC-NLP/MARBERT
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight    

In [7]:
# 7) جمع التقارير والنتائج وتنزيلها على الجهاز

import os, shutil
from google.colab import files

OUTPUT_DIR = "/content/outputs"

print("الملفات الموجودة داخل outputs:")
!find /content/outputs -type f

zip_path = "/content/reports_and_results"
shutil.make_archive(zip_path, "zip", OUTPUT_DIR)

files.download(zip_path + ".zip")

الملفات الموجودة داخل outputs:
/content/outputs/baseline_report_logreg.txt
/content/outputs/eda/04_top_tokens_not.png
/content/outputs/eda/04_top_tokens_s.png
/content/outputs/eda/01_class_balance.png
/content/outputs/eda/02_text_length_hist.png
/content/outputs/eda/05_dataset_overview.txt
/content/outputs/eda/03_text_length_box.png
/content/outputs/transformer_metrics.json
/content/outputs/comparison_table.csv
/content/outputs/baseline_report_svm.txt
/content/outputs/label_encoder.joblib
/content/outputs/svm.joblib
/content/outputs/baseline_metrics.json
/content/outputs/rf.joblib
/content/outputs/logreg.joblib
/content/outputs/baseline_metrics_rf.json
/content/outputs/transformer_report.txt
/content/outputs/baseline_report_rf.txt
/content/outputs/baseline_metrics_logreg.json
/content/outputs/plots/confusion_marbert.png
/content/outputs/plots/confusion_svm.png
/content/outputs/plots/confusion_logreg.png
/content/outputs/plots/confusion_rf.png
/content/outputs/plots/comparison_bars.png


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>